In [ ]:
import json
import urllib.parse
import boto3
import requests
from pprint import pprint

In [ ]:
# 1. Provide your long-term AWS programmatic credentials
AWS_ACCESS_KEY_ID = "YOUR_ACCESS_KEY_ID"
AWS_SECRET_ACCESS_KEY = "YOUR_SECRET_ACCESS_KEY"
SESSION_DURATION_SECONDS = 3600  # URL validity length (1 hour)

In [ ]:
print("[*] Contacting AWS STS to generate temporary credentials...")

try:
    # 2. Initialize the STS client using your credentials
    sts_client = boto3.client(
        "sts",
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    )

    # 3. Request temporary session tokens via GetFederationToken
    # Note: Your keys must have permission to call sts:GetFederationToken
    federation_response = sts_client.get_federation_token(
        Name="ConsoleFederatedUser", DurationSeconds=SESSION_DURATION_SECONDS
    )
    print("Federation response:")
    pprint(federation_response)

    credentials = federation_response["Credentials"]

    # 4. Format the session credentials into a JSON object
    #session_json = json.dumps(
    session_json = {
        "sessionId": credentials["AccessKeyId"],
        "sessionKey": credentials["SecretAccessKey"],
        "sessionToken": credentials["SessionToken"],
    }
    print("session_json:")
    pprint(session_json)
    encoded_session_json = urllib.parse.quote_plus(json.dumps(session_json))
    
    print("[*] Requesting a console sign-in token from the AWS federation endpoint...")

    # 5. Send a request to the AWS federation endpoint to get a sign-in token
    #federation_url = "https://amazon.com"
    federation_url = "https://ec2.us-east-1.amazonaws.com"
    #federation_url = "https://ec2.us-east-1.amazonaws.com/federation"
    token_request_params = {
        "Action": "getSigninToken",
        #"SessionDuration": SESSION_DURATION_SECONDS - 1,
        "Session": encoded_session_json
    }
    print("token request params:")
    pprint(token_request_params)
    token_response = requests.get(federation_url, params=token_request_params)
    #token_response = requests.post(federation_url, data=token_request_params)
    token_response.raise_for_status()
    print("token_response:")
    pprint(token_response.json())
    signin_token = token_response.json()["SigninToken"]

    # 6. Construct the final authenticated console URL
    # You can change the 'Destination' parameter to open a specific service page
    destination_url = "https://console.aws.amazon.com/"

    login_url_params = {
        "Action": "login",
        "Issuer": "CustomPythonScript",
        "Destination": destination_url,
        "SigninToken": signin_token,
    }

    final_login_url = f"{federation_url}?{urllib.parse.urlencode(login_url_params)}"

    print("\n[+] SUCCESS! Copy and paste the URL below into your browser:\n")
    print(final_login_url)

except sts_client.exceptions.ClientError as e:
    print(f"\n[!] AWS Error: {e.response['Error']['Message']}")
except Exception as e:
    print(f"\n[!] An unexpected error occurred: {e}")